# 06 — Final Holdout Evaluation (LHS 500)

Final generalization test using the 500-point LHS batch, untouched until now. Of 500 sampled cases, 491
converged in DWSIM (98.2%), consistent with the 98.6% convergence rate on the Sobol training batch.

Loading the already-fitted models and scalers saved from `05_Hyperparameter_Tuning.ipynb` rather than
retraining — guarantees this evaluation uses the exact same fitted objects, not a re-fit.

In [6]:
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import r2_score

input_columns = [
    "pressure_atm", "requested_vapor_fraction", "benzene_feed_fraction",
    "stages", "feed_stage_fraction", "reflux_ratio", "bottoms_fraction",
]
target_columns = ["x_D_benzene", "x_B_benzene", "Q_C", "Q_R"]

## Loading trained models and scalers

In [7]:
model_dir = "../models/01_trained_models_all_5"

rf_best = joblib.load(f"{model_dir}/random_forest.pkl")
xgb_best = joblib.load(f"{model_dir}/xgboost.pkl")
svr_best = joblib.load(f"{model_dir}/svr.pkl")
ann_best = joblib.load(f"{model_dir}/ann.pkl")
lin_final = joblib.load(f"{model_dir}/polynomial_regression.pkl")

scaler_X = joblib.load(f"{model_dir}/scaler_X.pkl")
scaler_y = joblib.load(f"{model_dir}/scaler_y.pkl")
poly = joblib.load(f"{model_dir}/poly_features.pkl")

## Loading the LHS holdout

In [8]:
lhs_df = pd.read_csv("../data/03_raw_holdout_lhs/lhs_500_converged.csv")

X_holdout = lhs_df[input_columns]
y_holdout = lhs_df[target_columns]

print("Holdout shape:", X_holdout.shape)

Holdout shape: (491, 7)


## Evaluating each model on the holdout — once

In [10]:
# Random Forest
pred = rf_best.predict(X_holdout)

holdout_results = {
    "RandomForest": {
        t: r2_score(y_holdout.iloc[:, i], pred[:, i])
        for i, t in enumerate(target_columns)
    }
}


# XGBoost
pred = xgb_best.predict(X_holdout)

holdout_results["XGBoost"] = {
    t: r2_score(y_holdout.iloc[:, i], pred[:, i])
    for i, t in enumerate(target_columns)
}


# SVR
X_holdout_scaled = scaler_X.transform(X_holdout)
pred = svr_best.predict(X_holdout_scaled)

holdout_results["SVR"] = {
    t: r2_score(y_holdout.iloc[:, i], pred[:, i])
    for i, t in enumerate(target_columns)
}


# ANN
pred_scaled = ann_best.predict(X_holdout_scaled)
pred = scaler_y.inverse_transform(pred_scaled)

holdout_results["ANN"] = {
    t: r2_score(y_holdout.iloc[:, i], pred[:, i])
    for i, t in enumerate(target_columns)
}


# Polynomial Regression
X_holdout_poly = poly.transform(X_holdout)
pred = lin_final.predict(X_holdout_poly)

holdout_results["PolynomialRegression"] = {
    t: r2_score(y_holdout.iloc[:, i], pred[:, i])
    for i, t in enumerate(target_columns)
}


holdout_df = pd.DataFrame(holdout_results).T

print("Holdout (LHS 500) R²:\n")
print(holdout_df.round(4))

Holdout (LHS 500) R²:

                      x_D_benzene  x_B_benzene     Q_C     Q_R
RandomForest               0.8638       0.7675  0.9774  0.9775
XGBoost                    0.9964       0.9967  0.9991  0.9991
SVR                        0.9935       0.9937  0.9951  0.9952
ANN                        0.9987       0.9985  0.9993  0.9993
PolynomialRegression       0.9822       0.9811  0.9997  0.9998


## Validation vs. Holdout comparison

A large gap between validation and holdout R² signals overfitting to the Sobol sampling pattern rather than
the underlying process.

In [11]:
val_df = pd.read_csv("../data/05_tuning_results/validation_r2_scores.csv", index_col="model")

comparison = pd.concat(
    {"Validation (Sobol 30%)": val_df, "Holdout (LHS 500)": holdout_df},
    axis=1
)
print(comparison.round(4))

drop_df = (val_df - holdout_df).round(4)
print("\nR² drop (validation → holdout), positive = performance dropped:\n")
print(drop_df)

                     Validation (Sobol 30%)                              \
                                x_D_benzene x_B_benzene     Q_C     Q_R   
RandomForest                         0.8464      0.7400  0.9712  0.9713   
XGBoost                              0.9954      0.9958  0.9990  0.9990   
SVR                                  0.9915      0.9921  0.9948  0.9949   
ANN                                  0.9984      0.9985  0.9993  0.9993   
PolynomialRegression                 0.9798      0.9789  0.9997  0.9998   

                     Holdout (LHS 500)                              
                           x_D_benzene x_B_benzene     Q_C     Q_R  
RandomForest                    0.8638      0.7675  0.9774  0.9775  
XGBoost                         0.9964      0.9967  0.9991  0.9991  
SVR                             0.9935      0.9937  0.9951  0.9952  
ANN                             0.9987      0.9985  0.9993  0.9993  
PolynomialRegression            0.9822      0.9811  0.9997  

### Reading the validation → holdout comparison

Every model scored equal to or better on the LHS holdout than on the Sobol validation split — all values in the
drop table are zero or negative, meaning no model overfit to the Sobol sampling pattern. This is a strong result:
it means the 7-input → 4-output relationships learned during training reflect real underlying process behavior,
not artifacts of how the training data happened to be sampled.

Random Forest shows the largest holdout improvement (x_D: +0.017, x_B: +0.027) but remains clearly the weakest
model overall — its accuracy ceiling, not its generalization, is the limiting factor.

XGBoost, SVR, ANN, and Polynomial Regression are all within 0.003 of their validation scores, and all above 0.98
on every output on the holdout. ANN is the strongest on x_D_benzene, Q_C, and Q_R; Polynomial Regression edges
it out specifically on Q_C/Q_R (0.9997/0.9998 vs ANN's 0.9993/0.9993).

Accuracy and robustness both point toward the same handful of models (XGBoost, SVR, ANN, Polynomial) as strong
candidates, with Random Forest ruled out. Final model selection still needs a physical consistency check — do
predictions stay within valid bounds (purity ∈ [0,1], duties ≥ 0) and follow expected monotonic trends — before
locking in a winner per output.